
# AMR 仿真数据生成器 — 完整使用手册

---

## 目录

```
1. 总体架构
2. 数据生成流水线（逐步图解）
3. 每个模块的原理详解
   3.1 星座映射
   3.2 RRC 脉冲成形
   3.3 频率调制（FSK/MSK）
   3.4 特殊调制（OQPSK/π4-DQPSK/2ASK）
   3.5 PA 非线性
   3.6 信道模型
   3.7 接收端损伤
4. 参数修改指南（含安全范围）
5. 常见问题与排错
6. 与训练脚本的接口规范
```

---

## 1. 总体架构

```
┌─────────────────────────────────────────────────────────────────┐
│                    generate_data_pro.py                         │
│                                                                 │
│  CONFIG 字典 ──→ 控制所有行为                                     │
│                                                                 │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐     │
│  │ 星座/调制  │ → │  PA(TX)   │ → │  信道     │ → │ 损伤(RX) │     │
│  │ 生成器    │   │  非线性   │   │ 衰落+噪声 │   │ 频偏/相噪│     │
│  └──────────┘   └──────────┘   └──────────┘   └──────────┘     │
│       ↓                                              ↓          │
│  (N, 1024) complex                        (N, 2, 1024) float32  │
│                                                      ↓          │
│                                            sionna_amr_data.npz  │
└─────────────────────────────────────────────────────────────────┘
```

输出文件结构：

```python
# sionna_amr_data.npz 内部：
{
    'X_train':   (N_train, 2, 1024)  float32,   # I/Q 两路信号
    'Y_train':   (N_train,)          int64,      # 类别索引 0~13
    'X_test':    (N_test, 2, 1024)   float32,
    'Y_test':    (N_test,)           int64,
    'snr_test':  (N_test,)           float32,    # 每个测试样本的 SNR
    'classes':   (14,)               str,        # 类别名称列表
    'snr_list':  (20,)               float32,    # SNR 档位列表
}
```

---

## 2. 数据生成流水线（逐步图解）

以生成一个 **16QAM, SNR=10dB** 的样本为例：

```
步骤 1: 随机比特
  [0,1,1,0, 1,0,0,1, 0,0,1,1, ...] × 256 个符号 × 4 bit/符号

步骤 2: 星座映射
  比特 → 复数符号: [-3+1j, 1+3j, -1-1j, ...]
  共 256 个复数点（NUM_SYMBOLS = 1024/4）

  Q ▲
    │  ×     ×     ×     ×
    │
    │  ×     ×     ×     ×     ← 16QAM: 4×4 方格
    │
    │  ×     ×     ×     ×
    │
    │  ×     ×     ×     ×
    └──────────────────────► I

步骤 3: 上采样 (×SPS=4)
  每个符号之间插入 3 个零
  [s0, 0, 0, 0, s1, 0, 0, 0, s2, 0, 0, 0, ...]
  长度: 256 × 4 = 1024

步骤 4: RRC 脉冲成形（FFT 卷积）
  ┌──────────────────────────────────────────────┐
  │  上采样信号 ──→ FFT ──→ × H(f) ──→ IFFT     │
  │                         ↑                     │
  │                    RRC 滤波器频谱              │
  └──────────────────────────────────────────────┘
  作用: 限制带宽，减少符号间干扰

步骤 5: PA 非线性（发射端）
  高功率放大器使信号幅度被压缩
  ┌──────────────────┐
  │  输入幅度 r       │
  │       ↓           │
  │  r_out = r / [1+(r/A_sat)^(2p)]^(1/2p)  │
  │       ↓           │
  │  外圈星座点被压向内圈                      │
  └──────────────────┘

步骤 6: 信道
  6a. 衰落（可选）:
      平坦 Rayleigh: y = h·x,  h ~ CN(0,1)
      多径: y = h0·x[n] + h1·x[n-1] + ... + h7·x[n-7]

  6b. AWGN 噪声:
      y = x + noise
      noise ~ CN(0, σ²),  σ² = 1/SNR_linear

步骤 7: 接收端损伤
  7a. 载波频偏:   y = x · exp(j·2π·Δf·n)        信号缓慢旋转
  7b. 相位噪声:   y = x · exp(j·φ[n])            φ 做随机游走
  7c. IQ不平衡:   I/Q 增益和相位不完全正交
  7d. DC偏移:     y = x + dc                     星座整体平移
  7e. 采样时偏:   采样点不在最佳位置

步骤 8: 转 IQ 格式
  complex (1024,) → real (2, 1024)
  第 0 行: I 路 (实部)
  第 1 行: Q 路 (虚部)

步骤 9: 保存
  所有样本拼接 → train/test 划分 → .npz 压缩存储
```

---

## 3. 每个模块的原理详解

### 3.1 星座映射

**目的**：将数字比特序列映射为复数平面上的点

```
调制方式       星座点数 M    每符号比特数    星座形状
──────────────────────────────────────────────────────
BPSK           2             1              实轴两点: {-1, +1}
QPSK           4             2              正方形 4 点
8PSK           8             3              圆周 8 等分点
16QAM          16            4              4×4 方格
32QAM          32            5              十字形（6×6 去 4 角）
64QAM          64            6              8×8 方格
128QAM         128           7              十字形（12×12 取 128 个）
256QAM         256           8              16×16 方格
```

**代码逻辑**：

```python
# 方形 QAM (4, 16, 64, 256):
def _make_square_qam(M):
    k = int(np.sqrt(M))              # 例如 16QAM → k=4
    levels = 2*np.arange(k) - (k-1)  # [-3, -1, 1, 3]
    gI, gQ = np.meshgrid(levels, levels)
    pts = (gI + 1j * gQ).flatten()   # 16 个复数点
    pts /= np.sqrt(np.mean(|pts|²))  # 归一化到单位平均功率
    return pts

# Cross-QAM (32, 128):
# 32QAM: 6×6=36 个点，去掉 4 个角 = 32
# 128QAM: 12×12=144 个点，按功率排序取最近 128 个

# PSK:
# M 个点均匀分布在单位圆上: exp(j·2πk/M), k=0,...,M-1

# 使用时: 从 M 个点中随机选 NUM_SYMBOLS 个
syms = constellation[np.random.randint(0, M, (batch, NUM_SYMBOLS))]
```

**为什么归一化到单位功率**：

```
不同阶数 QAM 原始功率不同:
  16QAM  原始平均功率 = 10
  256QAM 原始平均功率 = 170

归一化后都是 1.0
→ 公平比较不同调制在相同 SNR 下的性能
→ 功率归一化在训练时由 power_normalize() 再做一次
```

---

### 3.2 RRC 脉冲成形

**目的**：限制信号带宽，同时满足 Nyquist 零 ISI 条件

```
为什么需要脉冲成形？

不做:  [s0, 0, 0, 0, s1, 0, 0, 0, ...]
       ↓ 频谱无限宽，不现实

做了:  [s0 经过平滑滤波后的连续波形]
       ↓ 频谱被限制在有限带宽内
```

**RRC 滤波器参数**：

```python
design_rrc(ntaps=101, sps=4, beta=0.35)
```

```
ntaps (101): 滤波器长度，越长频率响应越陡
  ● 必须是奇数
  ● 太短 → 频率选择性差，有带外泄漏
  ● 太长 → 计算量大，时延增加
  ● 101 是实践中的常用值

sps (4): 每符号采样数
  ● 决定信号的过采样率
  ● SPS=4: 每个符号周期有 4 个采样点
  ● 符号数 = signal_length / sps = 1024/4 = 256

beta (0.35): 滚降系数 (Roll-off factor)
  ● 范围: [0, 1]
  ● β=0: 理想矩形频谱（不可实现）
  ● β=1: 最平滑，带宽最宽
  ● β=0.35: 3GPP LTE 推荐值
  ● 影响: 带宽 = (1+β)/(2·Ts)
```

**FFT 批量卷积（替代逐样本循环）**：

```python
# 传统（慢）:
for i in range(N):
    y[i] = np.convolve(x[i], rrc, mode='same')   # N 次卷积

# FFT 批量（快）:
X = FFT(所有 N 个信号)      # 一次 FFT
H = FFT(滤波器)             # 一次 FFT
Y = IFFT(X × H)             # 一次 IFFT
# 利用卷积定理: 时域卷积 = 频域乘积
```

---

### 3.3 频率调制（FSK/MSK）

**与 QAM/PSK 的本质区别**：

```
QAM/PSK: 信息在"星座点位置"（幅度/相位）
FSK:     信息在"瞬时频率"

FSK 不需要脉冲成形，直接生成时域波形
```

**CPFSK（连续相位频移键控）公式**：

```
s(t) = exp(j · φ(t))

φ(t) = 2π · h · Σ aₖ · ∫ g(τ-kT)dτ

简化（矩形脉冲）:
  在第 k 个符号周期内:
  瞬时频率 = h · aₖ / (2T)
  aₖ: 数据符号 (±1 或 ±1,±3)
  h:  调制指数
  T:  符号周期 = SPS 个采样点
```

**三种 FSK 的区别**：

```python
# 2FSK (h=1.0):
#   aₖ ∈ {-1, +1}
#   每符号相位变化 = ±π → 两个频率
delta_f = h / (2·SPS) = 1.0 / (2·4) = 0.125

# 4FSK (h=1.0):
#   aₖ ∈ {-3, -1, +1, +3}
#   四个频率等间隔
delta_f = 1.0 / (6·SPS) = 1.0 / 24 ≈ 0.0417

# MSK (h=0.5):
#   aₖ ∈ {-1, +1}
#   最小频移: 每符号相位变化 = ±π/2
delta_f = 0.5 / (2·SPS) = 0.5 / 8 = 0.0625
```

**矢量化实现**：

```python
def _gen_cpfsk(n, M, h_mod):
    syms = np.random.randint(0, M, (n, NUM_SYM))  # 随机符号索引
    levels = 2*np.arange(M) - (M-1)               # 符号值: [-1,1] 或 [-3,-1,1,3]
    freq_sym = levels[syms]                         # (N, NUM_SYM)

    # 每个采样点的频偏
    freq_per_sample = h_mod * freq_sym / (2*SPS)

    # 上采样: 每个符号值重复 SPS 次
    freq_up = np.repeat(freq_per_sample, SPS, axis=1)  # (N, SIG_LEN)

    # 累积相位 → 瞬时频率的积分 = 相位
    phase = 2π × cumsum(freq_up)

    return exp(j × phase)
```

---

### 3.4 特殊调制

#### OQPSK（偏移四相移键控）

```
与 QPSK 的区别：Q 路延迟半个符号周期

QPSK:    I ──┬── s0 ──┬── s1 ──┬── s2
         Q ──┘        └────────┘

OQPSK:   I ──┬── s0 ──┬── s1 ──┬── s2
         Q ────┬── s0 ──┬── s1 ──┬──
               ↑ 延迟 SPS/2 个样点

好处: 相位跳变最大 ±90°（QPSK 可能 ±180°）
     → 包络起伏更小 → 对 PA 更友好
```

```python
# I 路: 正常上采样位置
up_i[:, ::SPS] = bi                    # 0, 4, 8, 12, ...

# Q 路: 延迟 SPS//2 = 2 个样点
q_idx = np.arange(NUM_SYM) * SPS + SPS//2   # 2, 6, 10, 14, ...
up_q[:, q_idx] = bq
```

#### π/4-DQPSK（差分编码）

```
差分编码: 信息在相邻符号的相位差中

相位差 Δφ ∈ {π/4, 3π/4, -3π/4, -π/4}
对应 2 bit 数据

φ[k] = φ[k-1] + Δφ[k]    ← cumsum 实现
s[k] = exp(j·φ[k])
```

#### 2ASK（幅移键控 / OOK）

```
最简单的幅度调制:
  bit=0 → 幅度 0（不发）
  bit=1 → 幅度 1（发）

与 BPSK 的区别:
  BPSK: {-1, +1}   恒包络
  2ASK: {0, 1}     有零点
```

---

### 3.5 PA 非线性

**物理背景**：

```
发射机的功率放大器 (PA) 在高功率时会饱和

输入小信号 → 线性放大
输入大信号 → 增益压缩（削波）

对 QAM 的影响:
  内圈点: 线性区，不受影响
  外圈点: 被压缩向内 → 星座"缩水"
  
  64QAM 正常:              64QAM 经过 PA:
  ×  ×  ×  ×  ×  ×  ×  ×   ×  ×  ×  ×  ×  ×  ×  ×
  ×  ×  ×  ×  ×  ×  ×  ×    × ×  ×  ×  ×  ×  × ×
  ×  ×  ×  ×  ×  ×  ×  ×    × ×  ×  ×  ×  ×  × ×
  外圈间距 = 内圈间距       外圈间距 < 内圈间距
```

**Rapp SSPA 模型**：

```python
# AM/AM 特性（只影响幅度，不影响相位）
r_out = r_in / [1 + (r_in / A_sat)^(2p)]^(1/(2p))

# 参数:
# A_sat: 饱和幅度，由 IBO (Input Back-Off) 决定
#   A_sat = sqrt(P_avg × 10^(IBO/10))
#   IBO 越大 → A_sat 越大 → 越线性
#
# p: 平滑因子
#   p=1: 软限幅
#   p=3: 较陡的过渡（常用）
#   p→∞: 硬限幅
```

**IBO 参数的含义**：

```
IBO (Input Back-Off) = 饱和功率 / 平均输入功率 (dB)

IBO = 6 dB:  饱和点是平均功率的 4 倍 → 非线性明显
IBO = 10 dB: 饱和点是平均功率的 10 倍 → 轻微非线性
IBO = 15 dB: 饱和点是平均功率的 31 倍 → 几乎线性
IBO = 20 dB: 基本等于没有 PA 失真
```

---

### 3.6 信道模型

#### AWGN

```python
noise = sqrt(σ²/2) × (randn + j·randn)
y = x + noise

# σ² = 1/SNR_linear = 10^(-SNR_dB/10)
# 除以 2: 因为复噪声实部虚部各分一半功率
```

#### 平坦 Rayleigh 衰落

```
物理场景: 密集城区，无直射路径（NLOS）

h ~ CN(0, 1)  → 复高斯随机变量
y = h · x     → 整段信号乘同一个 h

|h| 服从 Rayleigh 分布:
  有时 |h| ≈ 1 → 正常
  有时 |h| ≈ 0 → 深度衰落（信号几乎消失）
```

#### 平坦 Rician 衰落

```
物理场景: 有直射路径（LOS）+ 散射

h = h_LOS + h_scatter

h_LOS = sqrt(K/(K+1)) · exp(jφ)     ← 确定性直射分量
h_scatter ~ CN(0, 1/(K+1))           ← 随机散射分量

K 因子:
  K → ∞: 纯直射，等于 AWGN
  K = 0:  纯散射，等于 Rayleigh
  K = 10 dB: 直射能量是散射的 10 倍
```

#### 多径频率选择性衰落

```
物理场景: 信号经过多条路径到达接收机，各路径时延不同

y[n] = h₀·x[n] + h₁·x[n-1] + h₂·x[n-2] + ... + h₇·x[n-7]

每条径:
  hₖ ~ CN(0, PDP[k])    独立衰落
  PDP[k] = exp(-k/τ_rms) 指数衰减功率时延谱

τ_rms (均方根时延扩展):
  小 → 时延差小 → 频率选择性弱
  大 → 时延差大 → 频率选择性强 → 某些频率深衰落

影响:
  ● 符号间干扰 (ISI): 前一个符号的"尾巴"影响当前符号
  ● 频率选择性: 不同频率衰落不同
  ● 比平坦衰落难处理得多
```

#### 混合信道

```python
'mixed_weights': {
    'awgn':           0.50,    # 50% 样本走纯 AWGN
    'flat_rayleigh':  0.20,    # 20% 走平坦 Rayleigh
    'flat_rician':    0.20,    # 20% 走平坦 Rician
    'multipath':      0.10,    # 10% 走多径
}

# 每个样本独立随机选择一种信道
# 目的: 让模型见过多种信道条件，提升泛化性
```

---

### 3.7 接收端损伤

施加顺序按真实硬件链路：

```
天线 → LNA → 混频器 → ADC → 数字信号
              ↓         ↓      ↓
           CFO/相噪   IQ不平衡  DC偏移/时偏
```

#### 3.7.1 载波频偏 (CFO)

```
原因: 发射机和接收机振荡器频率不完全相同

效果: y[n] = x[n] · exp(j·(2π·Δf·n + φ₀))
  ● 星座在复平面上缓慢旋转
  ● Δf 越大 → 旋转越快

Δf 的单位: 归一化频偏 = 实际频偏/采样率
  cfo_max_norm = 0.0003
  含义: 最大频偏 = 0.03% 的采样率
  例如: 采样率 1 MHz → 最大频偏 300 Hz
```

#### 3.7.2 相位噪声 (Phase Noise)

```
原因: 振荡器频率不是完美稳定的，有微小抖动

模型: Wiener 过程（随机游走）
  φ[n] = φ[n-1] + N(0, σ²)
  y[n] = x[n] · exp(j·φ[n])

效果: 星座点像"醉汉走路"一样随机偏移
  前面几个符号偏移小，后面越来越大

σ (phase_noise_std) 的影响:
  信号末端累积标准差 = σ × √L

  σ = 0.002, L = 1024:
    末端偏移 ≈ 0.002 × 32 = 0.064 rad ≈ 3.7°    ← 可接受

  σ = 0.01, L = 1024:
    末端偏移 ≈ 0.01 × 32 = 0.32 rad ≈ 18.3°     ← 太大！
    8PSK 符号间距才 45°，快分不清了
```

#### 3.7.3 IQ 不平衡

```
原因: 接收机混频器中 I/Q 两路不完美正交

模型:
  I_out = (1 + ε/2) · I_in
  Q_out = (1 - ε/2) · [Q_in·cos(Δφ) + I_in·sin(Δφ)]

  ε: 增益不平衡 (dB → 线性)
  Δφ: 相位不平衡 (度 → 弧度)

效果:
  正常 QPSK:          IQ 不平衡后:
    ×   ×                ×    ×
                    →         
    ×   ×              ×    ×
  (正方形)            (平行四边形)
```

#### 3.7.4 DC 偏移

```
原因: ADC 零点不在信号零点

效果: 星座整体平移
  y = x + (dc_I + j·dc_Q)

影响较小，但对 2ASK (OOK) 影响较大
（因为 2ASK 的一个星座点就在原点）
```

#### 3.7.5 采样时偏

```
原因: 接收机采样时钟与发射机不完全同步

效果: 采样点偏离最佳采样时刻

实现: 线性插值模拟分数延迟
  t_new = t + offset     (offset 是小数)
  y[t_new] = 线性插值(y[t], y[t+1])
```

---

## 4. 参数修改指南

### 4.1 核心参数速查表

```python
CONFIG = {
    # ┌──────────────────────────────────────────────────────────────┐
    # │ 参数名                  │ 默认值  │ 安全范围     │ 影响      │
    # ├──────────────────────────────────────────────────────────────┤
    # │ signal_length           │ 1024   │ 512~4096    │ 样本长度   │
    # │ sps                     │ 4      │ 2~8        │ 过采样率   │
    # │ rolloff                 │ 0.35   │ 0.2~0.5    │ 带宽      │
    # │ rrc_taps                │ 101    │ 51~201     │ 滤波器精度 │
    # │ n_per_class_per_snr     │ 200    │ 100~2000   │ 数据量    │
    # │ pa_ibo_db               │ 15     │ 8~30       │ PA失真    │
    # │ cfo_max_norm            │ 0.0003 │ 0~0.005    │ 频偏      │
    # │ phase_noise_std         │ 0.002  │ 0~0.005    │ 相噪      │
    # │ iq_gain_imb_max_db      │ 0.3    │ 0~2.0      │ IQ增益    │
    # │ iq_phase_imb_max_deg    │ 2.0    │ 0~10       │ IQ相位    │
    # │ dc_offset_max           │ 0.005  │ 0~0.05     │ DC偏移    │
    # │ timing_offset_max       │ 0.15   │ 0~0.5      │ 时偏      │
    # │ multipath_taps          │ 4      │ 2~16       │ 多径复杂度 │
    # │ multipath_tau_rms       │ 1.5    │ 0.5~5.0    │ 时延扩展  │
    # └──────────────────────────────────────────────────────────────┘
}
```

### 4.2 场景化配置模板

#### 场景 A：匹配 HisarMod 数据集

```python
# 目标: 仿真数据尽量接近 HisarMod，在其测试集上取高分
CONFIG_HISARMOD = {
    'signal_length':        1024,
    'sps':                  4,
    'rolloff':              0.35,
    'rrc_taps':             101,
    'n_per_class_per_snr':  200,
    'snr_list_db':          list(range(-18, 22, 2)),
    'train_ratio':          0.8,
    'save_dir':             './sionna_dataset',

    'channel_type':         'awgn',        # HisarMod 主要是 AWGN
    'pa_enable':            False,         # 关闭
    'impairments_enable':   False,         # 全部关闭

    'classes': ['BPSK', 'QPSK', '8PSK', 'OQPSK', '2FSK', '4FSK',
                '16QAM', '2ASK', 'Pi4DQPSK', '64QAM', '32QAM',
                '128QAM', '256QAM', 'MSK'],
    'seed': 42,
}
```

#### 场景 B：轻度真实化（推荐起点）

```python
# 目标: 比纯 AWGN 更真实，但不至于破坏信号可分性
CONFIG_MILD = {
    'signal_length':        1024,
    'sps':                  4,
    'rolloff':              0.35,
    'rrc_taps':             101,
    'n_per_class_per_snr':  300,           # 稍多一些
    'snr_list_db':          list(range(-18, 22, 2)),
    'train_ratio':          0.8,
    'save_dir':             './sionna_dataset',

    'channel_type':         'mixed',
    'mixed_weights':        {
        'awgn':             0.60,          # 60% 纯净
        'flat_rayleigh':    0.15,
        'flat_rician':      0.15,
        'multipath':        0.10,
    },
    'rician_k_db':          10.0,
    'multipath_taps':       4,
    'multipath_tau_rms':    1.0,

    'pa_enable':            True,
    'pa_ibo_db':            18.0,          # 很线性
    'pa_p':                 3,

    'impairments_enable':   True,
    'cfo_max_norm':         0.0002,
    'phase_noise_std':      0.001,         # 末端 ~1.8°
    'iq_gain_imb_max_db':   0.2,
    'iq_phase_imb_max_deg': 1.0,
    'dc_offset_max':        0.003,
    'timing_offset_max':    0.1,

    'classes': ['BPSK', 'QPSK', '8PSK', 'OQPSK', '2FSK', '4FSK',
                '16QAM', '2ASK', 'Pi4DQPSK', '64QAM', '32QAM',
                '128QAM', '256QAM', 'MSK'],
    'seed': 42,
}
```

#### 场景 C：真实无线环境

```python
# 目标: 模拟真实接收机，训练鲁棒的分类器
CONFIG_REALISTIC = {
    'signal_length':        1024,
    'sps':                  4,
    'rolloff':              0.35,
    'rrc_taps':             101,
    'n_per_class_per_snr':  500,           # 需要更多数据
    'snr_list_db':          list(range(-10, 22, 2)),  # 不用太低 SNR
    'train_ratio':          0.8,
    'save_dir':             './sionna_dataset',

    'channel_type':         'mixed',
    'mixed_weights':        {
        'awgn':             0.30,
        'flat_rayleigh':    0.25,
        'flat_rician':      0.25,
        'multipath':        0.20,
    },
    'rician_k_db':          8.0,
    'multipath_taps':       6,
    'multipath_tau_rms':    2.0,

    'pa_enable':            True,
    'pa_ibo_db':            12.0,
    'pa_p':                 3,

    'impairments_enable':   True,
    'cfo_max_norm':         0.0005,
    'phase_noise_std':      0.003,         # 末端 ~5.5°
    'iq_gain_imb_max_db':   0.5,
    'iq_phase_imb_max_deg': 3.0,
    'dc_offset_max':        0.01,
    'timing_offset_max':    0.2,

    'classes': ['BPSK', 'QPSK', '8PSK', 'OQPSK', '2FSK', '4FSK',
                '16QAM', '2ASK', 'Pi4DQPSK', '64QAM', '32QAM',
                '128QAM', '256QAM', 'MSK'],
    'seed': 42,
}
```

### 4.3 单参数调节效果预测

```
想提高准确率 → 降低难度:
  ● pa_ibo_db ↑         (15 → 20)
  ● phase_noise_std ↓   (0.002 → 0.001)
  ● multipath 比例 ↓    (0.10 → 0.05)
  ● multipath_taps ↓    (4 → 2)
  ● cfo_max_norm ↓      (0.0003 → 0.0001)

想提高鲁棒性 → 增加难度:
  ● pa_ibo_db ↓         (15 → 10)
  ● mixed 中 AWGN 比例 ↓
  ● n_per_class_per_snr ↑ (同时增加数据量)

想让高阶 QAM 更好分:
  ● pa_enable = False    (PA 对高阶 QAM 伤害最大)
  ● phase_noise_std ↓    (相噪对密集星座伤害大)
  ● iq 参数 ↓            (IQ 不平衡扭曲星座)

想让 FSK 更好分:
  ● cfo_max_norm ↓       (频偏直接干扰频率特征)
  ● phase_noise_std ↓    (相噪模糊瞬时频率)
```

### 4.4 参数之间的约束关系

```
约束 1:  signal_length 必须能被 sps 整除
         1024 / 4 = 256 ✓
         1024 / 3 = 341.3 ✗

约束 2:  rrc_taps 必须是奇数
         101 ✓    100 ✗

约束 3:  phase_noise_std × √signal_length < 最小星座间距/3
         对于 8PSK: 间距 = π/4 ≈ 0.785 rad
         要求: σ × √1024 < 0.785/3 ≈ 0.26
         即: σ < 0.008

约束 4:  multipath_taps × multipath_tau_rms 不宜 > signal_length/10
         否则信道扩展太长，信号严重失真
         4 × 1.5 = 6 << 102 ✓

约束 5:  n_per_class_per_snr × len(snr_list) × 14 = 总样本数
         200 × 20 × 14 = 56,000
         建议总样本不少于 30,000（训练集 ~24,000）
```

---

## 5. 常见问题与排错

### Q1: 准确率太低（<70%）

```
检查清单:
  □ phase_noise_std 是否太大？ → 降到 0.002 以下
  □ pa_ibo_db 是否太小？ → 升到 15 以上
  □ multipath 比例是否太高？ → 降到 10% 以下
  □ 是否所有损伤都开着？ → 先全关，逐个开
```

### Q2: 高阶 QAM (128/256) 互相混淆

```
原因: 星座密度接近，任何失真都让它们不可区分
解决:
  ● 关闭 PA（pa_enable=False）
  ● 降低相位噪声（phase_noise_std < 0.001）
  ● 降低 IQ 不平衡
  ● 或接受它们就是难分的
```

### Q3: 训练集准确率高但测试集低

```
原因: 过拟合
解决:
  ● 增加 n_per_class_per_snr (200 → 500)
  ● 训练时 augment=True 已开启
  ● 检查模型是否太大
```

### Q4: 生成速度太慢

```
瓶颈通常在:
  ● timing_offset: 有 for 循环 → 可关闭（影响最小的损伤）
  ● multipath: 有 for 循环 → 减少 taps 数
  ● 样本数太多 → 减少 n_per_class_per_snr
```

### Q5: 内存不够

```
56,000 × 2 × 1024 × 4 bytes = ~437 MB
如果增加到 500 per class per snr:
  140,000 × 2 × 1024 × 4 = ~1.1 GB

解决:
  ● 减少 n_per_class_per_snr
  ● 减少 SNR 档位数
  ● 使用 float16 存储（np.float16）
```

---

## 6. 与训练脚本的接口规范

### 数据格式约定

```python
# 生成器输出 → 训练器输入，必须满足:

X_train.shape = (N_train, 2, 1024)     # 第 0 维: I路, 第 1 维: Q路
X_train.dtype = np.float32

Y_train.shape = (N_train,)
Y_train.dtype = np.int64               # 0 ~ 13

classes 顺序必须一致:
  ['BPSK','QPSK','8PSK','OQPSK','2FSK','4FSK',
   '16QAM','2ASK','Pi4DQPSK','64QAM','32QAM',
   '128QAM','256QAM','MSK']

snr_test.shape = (N_test,)             # 标量，不是 one-hot
```

### 训练脚本加载代码

```python
# 这段代码在任何配置下都不需要改:
DATA_PATH = './sionna_dataset/sionna_amr_data.npz'
data = np.load(DATA_PATH, allow_pickle=True)

classes          = list(data['classes'])
train_raw        = data['X_train']
train_labels_idx = data['Y_train']
test_raw         = data['X_test']
test_labels_idx  = data['Y_test']
Y_test_onehot    = np.eye(len(classes), dtype='uint8')[test_labels_idx]
Z_test           = data['snr_test'].reshape(-1, 1)
```

### 如果修改了 signal_length 或类别数

```python
# signal_length 改了 → 模型也要改:
# model_tcn.py / model_transformer.py 中的输入长度参数

# 类别数改了 → build_model(classes=新数量)
# 训练脚本自动适配: build_model(classes=len(classes))
```



# 三个问题的详细解答

## 问题 1：每 4 个比特构成一个符号？

**是的，完全正确。**

```
16QAM: 每符号 log₂(16) = 4 个比特

比特流: [0,1,1,0, 1,0,0,1, 0,0,1,1, 1,1,0,0, ...]
         ├──────┤ ├──────┤ ├──────┤ ├──────┤
         符号 1    符号 2    符号 3    符号 4    ...

总比特数 = 256 符号 × 4 比特/符号 = 1024 比特
总符号数 = signal_length / sps = 1024 / 4 = 256 个符号
```

其他调制的对应关系：

```
调制方式    M     每符号比特数    256个符号需要多少比特
────────────────────────────────────────────────────
BPSK        2     1              256
QPSK        4     2              512
8PSK        8     3              768
16QAM      16     4              1024
32QAM      32     5              1280
64QAM      64     6              1536
128QAM    128     7              1792
256QAM    256     8              2048
```

---

## 问题 2：比特到复数符号怎么映射的？

以 **16QAM** 为例，用格雷码映射（相邻点只差 1 bit）：

```
4 个比特 = (b₃, b₂, b₁, b₀)

拆成两组:
  I 路由 (b₃, b₂) 决定
  Q 路由 (b₁, b₀) 决定

每组 2 bit → 4 个电平值:
  (0,0) → -3
  (0,1) → -1      ← 格雷码: 相邻值只差 1 bit
  (1,1) → +1
  (1,0) → +3

完整映射表:
  (b₃,b₂) → I      (b₁,b₀) → Q      复数符号
  ─────────────────────────────────────────
  (0,0)   → -3     (0,0)   → -3     -3 - 3j
  (0,0)   → -3     (0,1)   → -1     -3 - 1j
  (0,0)   → -3     (1,1)   → +1     -3 + 1j
  (0,0)   → -3     (1,0)   → +3     -3 + 3j
  (0,1)   → -1     (0,0)   → -3     -1 - 3j
  ...              ...               ...
```

**具体例子**：

```
比特 (0,1,1,0):
  I 路: (0,1) → -1
  Q 路: (1,0) → +3
  复数符号: -1 + 3j

比特 (1,0,0,1):
  I 路: (1,0) → +3
  Q 路: (0,1) → -1
  复数符号: +3 - 1j

比特 (0,0,1,1):
  I 路: (0,0) → -3
  Q 路: (1,1) → +1
  复数符号: -3 + 1j
```

在复平面上的位置：

```
Q
+3 │  ×         ×         ×         ×
   │ (-3+3j)  (-1+3j)  (+1+3j)  (+3+3j)
   │  0010     0110      1110     1010
   │
+1 │  ×         ×         ×         ×
   │ (-3+1j)  (-1+1j)  (+1+1j)  (+3+1j)
   │  0011     0111      1111     1011
   │
-1 │  ×         ×         ×         ×
   │ (-3-1j)  (-1-1j)  (+1-1j)  (+3-1j)
   │  0001     0101      1101     1001
   │
-3 │  ×         ×         ×         ×
   │ (-3-3j)  (-1-3j)  (+1-3j)  (+3-3j)
   │  0000     0100      1100     1000
   │
   └──────────────────────────────────── I
      -3        -1        +1        +3
```

**注意格雷码特性**：任意相邻两个点只差 1 个 bit，这样判错到邻近点时只错 1 bit 而不是多 bit。

---

## 问题 3：4×4 方格确实是 I[-3,-1,1,3] × Q[-3,-1,1,3]

**完全正确。** 代码中对应的实现：

```python
def _make_square_qam(M):
    k = int(np.sqrt(M))               # M=16 → k=4
    levels = 2 * np.arange(k) - (k-1) # 2×[0,1,2,3] - 3 = [-3,-1,1,3]
    gI, gQ = np.meshgrid(levels, levels)
    pts = (gI + 1j * gQ).flatten()     # 4×4=16 个复数
    pts /= np.sqrt(np.mean(np.abs(pts)**2))  # 归一化
    return pts
```

逐步展开：

```python
k = 4
np.arange(k)           = [0, 1, 2, 3]
2 * np.arange(k)       = [0, 2, 4, 6]
2 * np.arange(k) - 3   = [-3, -1, 1, 3]    ← 就是 I 和 Q 的电平值

# meshgrid 生成所有组合:
gI = [[-3, -1,  1,  3],       gQ = [[-3, -3, -3, -3],
      [-3, -1,  1,  3],             [-1, -1, -1, -1],
      [-3, -1,  1,  3],             [ 1,  1,  1,  1],
      [-3, -1,  1,  3]]             [ 3,  3,  3,  3]]

# 合成复数:
gI + j·gQ = [[-3-3j, -1-3j,  1-3j,  3-3j],
             [-3-1j, -1-1j,  1-1j,  3-1j],
             [-3+1j, -1+1j,  1+1j,  3+1j],
             [-3+3j, -1+3j,  1+3j,  3+3j]]

# flatten → 16 个复数点
# 归一化: 平均功率 = mean(|-3|²+|-1|²+|1|²+|3|²) × 2 = mean(9+1+1+9)×2 = 10
# pts /= √10 → 单位平均功率
```

其他阶数的电平值：

```
调制       k    levels                          点数
──────────────────────────────────────────────────────
QPSK(4QAM) 2    [-1, 1]                         2×2=4
16QAM      4    [-3, -1, 1, 3]                  4×4=16
64QAM      8    [-7,-5,-3,-1,1,3,5,7]           8×8=64
256QAM    16    [-15,-13,...,-1,1,...,13,15]     16×16=256
```

---

## 补充：代码中的简化

当前代码**没有实现格雷码映射**，而是直接随机选星座点：

```python
# 当前代码（简化版）:
def _rand_symbols(name, n_batch, n_sym):
    c = CONSTELLATIONS[name]           # 16 个预生成的复数点
    return c[np.random.randint(0, 16, (n_batch, n_sym))]
    #        ↑ 直接随机选索引，跳过了"比特→格雷码→星座点"

# 严格版（如果需要）:
def _rand_symbols_gray(M, n_batch, n_sym):
    bits = np.random.randint(0, 2, (n_batch, n_sym, int(np.log2(M))))
    # bits → 格雷码索引 → 星座点
    ...
```

**对 AMR 任务来说没有区别**，因为分类器看的是波形形状，不关心哪个比特映射到哪个点。随机选点和格雷码选点生成的信号统计特性完全相同。